# HieraCascadeNet — Full Reproducible Pipeline

**A Dual-Backbone Cross-Modal Fusion Framework with Cascade Attention Decoding for Joint Skin-Lesion Segmentation and Classification**

This notebook is an end-to-end PyTorch implementation of the architecture described in the paper. It covers:

1. **Data** — HAM10000 loading, near-duplicate filtering, stratified split, optional lesion masks.
2. **Preprocessing & augmentation** — resize/normalize + train-only geometric aug.
3. **Class-balanced oversampling** — every class balanced to the majority count *after* the split, on the training portion only.
4. **Model** — EfficientNet-B4 (CNN) + Swin-Base (Transformer) dual encoders → **Cross-Modal Feature-Fusion Gate (CMFG)** → **cascade attention decoder** (CBAM + attention-gated skips + ASPP + strip-pooling bottleneck) → segmentation head + 7-class classification head + deep supervision.
5. **Loss** — `L_total = L_cls + λ1·L_seg + λ2·L_ds` with Dice-Focal / focal-CE.
6. **Two-stage training** — freeze encoders → adapt fusion/decoder/heads, then unfreeze and fine-tune end-to-end (AdamW + cosine LR + grad clip + early stopping).
7. **Evaluation** — accuracy, macro P/R/F1, per-class F1, confusion matrix, one-vs-rest ROC + AUC, precision–recall + AP.
8. **Interpretability** — Grad-CAM, backbone feature maps, t-SNE of penultimate embeddings.
9. **Computational cost** — params, FLOPs, latency, throughput.

> **Note on reproducing exact numbers.** The paper reports 93.57% accuracy / 0.9356 macro-F1 on a class-balanced 4,694-image test set. This notebook reproduces the *method and pipeline*; exact figures depend on the HAM10000 download, mask availability, seeds, and hardware. Treat the config below as a faithful starting point and tune as needed.

---


## 0. Environment setup

Install dependencies. On Colab/Kaggle run this cell once; on a configured machine you can skip it.

In [ ]:
# If needed, uncomment to install.
# !pip install -q torch torchvision timm albumentations opencv-python-headless \
#     scikit-learn pandas numpy matplotlib seaborn tqdm ptflops grad-cam

import os, sys, math, random, time, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import timm
print("torch", torch.__version__, "| timm", timm.__version__)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

In [ ]:
def seed_everything(seed: int = 42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

seed_everything(42)

## 1. Configuration

All hyperparameters in one place. The seven HAM10000 diagnostic classes and the two-stage schedule follow the paper (frozen adaptation → end-to-end fine-tuning, AdamW + cosine LR, grad clip 1.0, early stopping patience 15).

In [ ]:
from dataclasses import dataclass, field
from typing import List, Optional

# HAM10000 diagnostic classes (order fixed for the whole pipeline)
CLASSES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)

# Human-readable names (for plots)
CLASS_FULL = {
    "akiec": "Actinic keratoses / intraepithelial carcinoma",
    "bcc":   "Basal cell carcinoma",
    "bkl":   "Benign keratosis-like lesions",
    "df":    "Dermatofibroma",
    "mel":   "Melanoma",
    "nv":    "Melanocytic nevi",
    "vasc":  "Vascular lesions",
}

@dataclass
class CFG:
    # ---- paths (edit to your environment) ----
    data_root: str = "./data/ham10000"          # folder containing the images + metadata
    metadata_csv: str = "HAM10000_metadata.csv"   # standard HAM10000 metadata file
    image_dirs: List[str] = field(default_factory=lambda: ["HAM10000_images_part_1",
                                                            "HAM10000_images_part_2"])
    mask_dir: Optional[str] = "HAM10000_segmentations_lesion_tschandl"  # optional; None if unavailable
    out_dir: str = "./outputs"

    # ---- image ----
    img_size: int = 224
    in_chans: int = 3

    # ---- backbones (timm names) ----
    cnn_backbone: str = "efficientnet_b4"
    swin_backbone: str = "swin_base_patch4_window7_224"
    fused_dims: List[int] = field(default_factory=lambda: [128, 256, 512, 1024])  # shared dims per stage
    decoder_dim: int = 512   # bottleneck output channels

    # ---- split ----
    val_frac: float = 0.15
    test_frac: float = 0.20   # paper test set ~4,694 imgs; stratified
    seed: int = 42
    dedup: bool = True        # drop near-duplicate lesion_ids across splits

    # ---- training ----
    batch_size: int = 16
    num_workers: int = 4
    epochs_frozen: int = 30      # stage 1: encoders frozen
    epochs_finetune: int = 60    # stage 2: end-to-end
    lr_frozen: float = 1e-3
    lr_finetune: float = 1e-5    # lower LR when encoders unfrozen (paper: 1e-4 head / 1e-5 backbone)
    lr_backbone: float = 1e-5
    weight_decay: float = 1e-4
    grad_clip: float = 1.0
    early_stop_patience: int = 15
    amp: bool = True

    # ---- loss weights: L_total = L_cls + λ1 L_seg + λ2 L_ds ----
    lambda_seg: float = 0.4
    lambda_ds:  float = 0.3
    focal_gamma: float = 2.0
    label_smoothing: float = 0.05
    use_class_balanced_sampler: bool = True

cfg = CFG()
os.makedirs(cfg.out_dir, exist_ok=True)
cfg

## 2. Build the data index

We read `HAM10000_metadata.csv`, resolve each image path across the two image folders, and (optionally) attach a lesion mask path. HAM10000 ships without masks by default; the community `HAM10000_segmentations_lesion_tschandl` set provides them. If masks are missing, the pipeline still runs — the segmentation branch simply trains on whatever masks are present and the classification path is unaffected.

Near-duplicate handling follows the paper: HAM10000 contains multiple views of the same `lesion_id`, so we keep all rows for a lesion **inside a single split** to prevent content leakage across train/val/test.

In [ ]:
def build_dataframe(cfg: CFG) -> pd.DataFrame:
    root = Path(cfg.data_root)
    meta = pd.read_csv(root / cfg.metadata_csv)
    # locate each image
    search_dirs = [root / d for d in cfg.image_dirs] + [root]
    def find_img(image_id):
        for d in search_dirs:
            p = d / f"{image_id}.jpg"
            if p.exists():
                return str(p)
        return None
    meta["img_path"] = meta["image_id"].map(find_img)
    missing = meta["img_path"].isna().sum()
    if missing:
        print(f"[warn] {missing} images not found on disk; dropping them.")
        meta = meta.dropna(subset=["img_path"]).reset_index(drop=True)

    # optional masks
    if cfg.mask_dir is not None and (root / cfg.mask_dir).exists():
        mdir = root / cfg.mask_dir
        def find_mask(image_id):
            for suffix in (f"{image_id}_segmentation.png", f"{image_id}.png"):
                p = mdir / suffix
                if p.exists():
                    return str(p)
            return None
        meta["mask_path"] = meta["image_id"].map(find_mask)
        n_masks = meta["mask_path"].notna().sum()
        print(f"[info] found {n_masks}/{len(meta)} lesion masks.")
    else:
        meta["mask_path"] = None
        print("[info] no mask directory — segmentation branch trains only where masks exist (none here).")

    meta["label"] = meta["dx"].map(CLASS_TO_IDX)
    meta = meta.dropna(subset=["label"]).reset_index(drop=True)
    meta["label"] = meta["label"].astype(int)
    return meta

# NOTE: this cell requires the dataset on disk. If you don't have it yet,
# skip to the "synthetic smoke test" cell at the bottom to validate the model wiring.
try:
    df = build_dataframe(cfg)
    print(df["dx"].value_counts())
    DATA_AVAILABLE = True
except FileNotFoundError as e:
    print("[info] dataset not found — set cfg.data_root. You can still run the model smoke test.")
    df, DATA_AVAILABLE = None, False

## 3. Stratified, leakage-safe split

Stratify on the diagnosis so class proportions stay consistent across train/val/test, while grouping by `lesion_id` so that all views of one lesion land in the same split.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

def stratified_group_split(df, cfg):
    # per-lesion label (mode) for stratification
    lesion_lab = df.groupby("lesion_id")["label"].agg(lambda s: s.mode().iloc[0])
    groups = df["lesion_id"].values
    y = df["label"].values

    # test split
    sgkf = StratifiedGroupKFold(n_splits=int(round(1/cfg.test_frac)),
                                shuffle=True, random_state=cfg.seed)
    trainval_idx, test_idx = next(sgkf.split(df, y, groups))
    df_tv, df_test = df.iloc[trainval_idx].reset_index(drop=True), df.iloc[test_idx].reset_index(drop=True)

    # val split out of trainval
    rel_val = cfg.val_frac / (1 - cfg.test_frac)
    sgkf2 = StratifiedGroupKFold(n_splits=int(round(1/rel_val)),
                                 shuffle=True, random_state=cfg.seed)
    tr_idx, val_idx = next(sgkf2.split(df_tv, df_tv["label"].values, df_tv["lesion_id"].values))
    df_train, df_val = df_tv.iloc[tr_idx].reset_index(drop=True), df_tv.iloc[val_idx].reset_index(drop=True)
    return df_train, df_val, df_test

if DATA_AVAILABLE:
    df_train, df_val, df_test = stratified_group_split(df, cfg)
    print("train/val/test:", len(df_train), len(df_val), len(df_test))
    # sanity: no lesion_id leakage
    for a,b in [("train","val"),("train","test"),("val","test")]:
        overlap = set(eval(f"df_{a}").lesion_id) & set(eval(f"df_{b}").lesion_id)
        assert not overlap, f"leak between {a}/{b}!"
    print("no lesion_id leakage across splits ✓")

## 4. Preprocessing, augmentation & Dataset

Train images get random rotations, H/V flips, mild zoom and small shifts (dermoscopy has no canonical orientation); val/test are only resized + normalized. ImageNet statistics are used because both backbones are ImageNet-pretrained.

In [ ]:
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

def train_tf(sz):
    return A.Compose([
        A.Resize(sz, sz),
        A.Rotate(limit=25, border_mode=cv2.BORDER_REFLECT_101, p=0.7),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.06, scale_limit=0.12, rotate_limit=0,
                           border_mode=cv2.BORDER_REFLECT_101, p=0.5),
        A.RandomBrightnessContrast(0.1, 0.1, p=0.3),
        A.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ToTensorV2(),
    ], additional_targets={"mask": "mask"})

def eval_tf(sz):
    return A.Compose([
        A.Resize(sz, sz),
        A.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ToTensorV2(),
    ], additional_targets={"mask": "mask"})


class HAMDataset(Dataset):
    def __init__(self, frame, tf, img_size):
        self.df = frame.reset_index(drop=True)
        self.tf = tf
        self.sz = img_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = cv2.cvtColor(cv2.imread(row["img_path"]), cv2.COLOR_BGR2RGB)

        mp = row.get("mask_path", None)
        has_mask = isinstance(mp, str) and os.path.exists(mp)
        if has_mask:
            m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
            m = (m > 127).astype(np.float32)
        else:
            m = np.zeros(img.shape[:2], dtype=np.float32)

        out = self.tf(image=img, mask=m)
        image = out["image"]
        mask = out["mask"].unsqueeze(0).float()  # (1,H,W)
        return {
            "image": image,
            "mask": mask,
            "label": torch.tensor(int(row["label"]), dtype=torch.long),
            "has_mask": torch.tensor(1.0 if has_mask else 0.0),
        }

## 5. Class-balanced oversampling

The paper balances every class to the majority count (6,705) *on the training portion only*, after the split. We implement this with a `WeightedRandomSampler` giving inverse-frequency weights, so each epoch draws classes with (approximately) equal probability — no leakage, no test contamination.

In [ ]:
from torch.utils.data import WeightedRandomSampler, RandomSampler

def make_balanced_sampler(frame):
    labels = frame["label"].values
    counts = np.bincount(labels, minlength=NUM_CLASSES).astype(np.float64)
    counts[counts == 0] = 1.0
    class_w = 1.0 / counts
    sample_w = class_w[labels]
    # length = balanced dataset size (majority * num_classes), like the paper's 6705*7 target
    length = int(counts.max() * NUM_CLASSES)
    return WeightedRandomSampler(torch.as_tensor(sample_w, dtype=torch.double),
                                 num_samples=length, replacement=True)

def make_loaders(cfg, df_train, df_val, df_test):
    ds_tr = HAMDataset(df_train, train_tf(cfg.img_size), cfg.img_size)
    ds_va = HAMDataset(df_val,   eval_tf(cfg.img_size),  cfg.img_size)
    ds_te = HAMDataset(df_test,  eval_tf(cfg.img_size),  cfg.img_size)

    if cfg.use_class_balanced_sampler:
        sampler = make_balanced_sampler(df_train)
        dl_tr = DataLoader(ds_tr, batch_size=cfg.batch_size, sampler=sampler,
                           num_workers=cfg.num_workers, pin_memory=True, drop_last=True)
    else:
        dl_tr = DataLoader(ds_tr, batch_size=cfg.batch_size, shuffle=True,
                           num_workers=cfg.num_workers, pin_memory=True, drop_last=True)
    dl_va = DataLoader(ds_va, batch_size=cfg.batch_size, shuffle=False,
                       num_workers=cfg.num_workers, pin_memory=True)
    dl_te = DataLoader(ds_te, batch_size=cfg.batch_size, shuffle=False,
                       num_workers=cfg.num_workers, pin_memory=True)
    return dl_tr, dl_va, dl_te

if DATA_AVAILABLE:
    dl_train, dl_val, dl_test = make_loaders(cfg, df_train, df_val, df_test)
    print("batches:", len(dl_train), len(dl_val), len(dl_test))

## 6. Model — building blocks

We implement the attention/pooling primitives referenced in the paper:

- **CBAM** — channel attention (avg+max pooled MLP) followed by spatial attention (7×7 conv on channel-pooled maps).
- **Attention gate** — the Attention U-Net gating unit that suppresses irrelevant background on the skip path.
- **ASPP** — atrous spatial pyramid pooling for isotropic multi-scale context (rates 6/12/18 + image pooling).
- **Strip pooling** — long, narrow horizontal & vertical pooling kernels for elongated/anisotropic structures.


In [ ]:
class ChannelAttention(nn.Module):
    def __init__(self, c, r=16):
        super().__init__()
        h = max(c // r, 8)
        self.mlp = nn.Sequential(nn.Conv2d(c, h, 1, bias=False), nn.ReLU(inplace=True),
                                 nn.Conv2d(h, c, 1, bias=False))
    def forward(self, x):
        a = self.mlp(F.adaptive_avg_pool2d(x, 1))
        m = self.mlp(F.adaptive_max_pool2d(x, 1))
        return torch.sigmoid(a + m)

class SpatialAttention(nn.Module):
    def __init__(self, k=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, k, padding=k // 2, bias=False)
    def forward(self, x):
        avg = x.mean(dim=1, keepdim=True)
        mx, _ = x.max(dim=1, keepdim=True)
        return torch.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))

class CBAM(nn.Module):
    def __init__(self, c, r=16, k=7):
        super().__init__()
        self.ca = ChannelAttention(c, r)
        self.sa = SpatialAttention(k)
    def forward(self, x):
        x = x * self.ca(x)
        x = x * self.sa(x)
        return x

class AttentionGate(nn.Module):
    """Attention U-Net gate: g=gating (decoder), x=skip (encoder)."""
    def __init__(self, in_g, in_x, inter):
        super().__init__()
        self.Wg = nn.Sequential(nn.Conv2d(in_g, inter, 1, bias=True), nn.BatchNorm2d(inter))
        self.Wx = nn.Sequential(nn.Conv2d(in_x, inter, 1, bias=True), nn.BatchNorm2d(inter))
        self.psi = nn.Sequential(nn.Conv2d(inter, 1, 1, bias=True), nn.BatchNorm2d(1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)
    def forward(self, g, x):
        if g.shape[-2:] != x.shape[-2:]:
            g = F.interpolate(g, size=x.shape[-2:], mode="bilinear", align_corners=False)
        a = self.psi(self.relu(self.Wg(g) + self.Wx(x)))
        return x * a

class ASPP(nn.Module):
    def __init__(self, cin, cout, rates=(6, 12, 18)):
        super().__init__()
        self.b0 = nn.Sequential(nn.Conv2d(cin, cout, 1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))
        self.br = nn.ModuleList([
            nn.Sequential(nn.Conv2d(cin, cout, 3, padding=r, dilation=r, bias=False),
                          nn.BatchNorm2d(cout), nn.ReLU(inplace=True)) for r in rates])
        self.pool = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(cin, cout, 1, bias=False),
                                  nn.BatchNorm2d(cout), nn.ReLU(inplace=True))
        self.proj = nn.Sequential(nn.Conv2d(cout * (len(rates) + 2), cout, 1, bias=False),
                                  nn.BatchNorm2d(cout), nn.ReLU(inplace=True))
    def forward(self, x):
        feats = [self.b0(x)] + [b(x) for b in self.br]
        p = self.pool(x)
        p = F.interpolate(p, size=x.shape[-2:], mode="bilinear", align_corners=False)
        feats.append(p)
        return self.proj(torch.cat(feats, dim=1))

class StripPooling(nn.Module):
    """Horizontal + vertical strip pooling (Hou et al., 2020)."""
    def __init__(self, c):
        super().__init__()
        self.ph = nn.AdaptiveAvgPool2d((1, None))   # pool over height
        self.pv = nn.AdaptiveAvgPool2d((None, 1))   # pool over width
        self.ch = nn.Conv2d(c, c, (1, 3), padding=(0, 1), bias=False)
        self.cv = nn.Conv2d(c, c, (3, 1), padding=(1, 0), bias=False)
        self.fuse = nn.Sequential(nn.Conv2d(c, c, 1, bias=False), nn.BatchNorm2d(c))
    def forward(self, x):
        H, W = x.shape[-2:]
        h = F.interpolate(self.ch(self.ph(x)), size=(H, W), mode="bilinear", align_corners=False)
        v = F.interpolate(self.cv(self.pv(x)), size=(H, W), mode="bilinear", align_corners=False)
        return x * torch.sigmoid(self.fuse(h + v))


## 7. Cross-Modal Feature-Fusion Gate (CMFG)

Rather than concatenating the two streams, the CMFG projects the CNN (texture) and Swin (context) features to a shared dimensionality at each stage, then lets one modality **gate** the other. A learned gate `α = σ(conv([c, s]))` mixes them: `fused = α·c + (1−α)·s`, followed by a residual conv. This lets texture-rich and context-rich evidence reinforce each other instead of getting averaged into noise.

In [ ]:
class CrossModalFusionGate(nn.Module):
    def __init__(self, c_cnn, c_swin, c_out):
        super().__init__()
        self.pc = nn.Sequential(nn.Conv2d(c_cnn, c_out, 1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True))
        self.ps = nn.Sequential(nn.Conv2d(c_swin, c_out, 1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True))
        # gate conditioned on both modalities
        self.gate = nn.Sequential(nn.Conv2d(2 * c_out, c_out, 1, bias=False),
                                  nn.BatchNorm2d(c_out), nn.Sigmoid())
        # cross modulation: each modality modulates the other
        self.mod_c = nn.Sequential(nn.Conv2d(c_out, c_out, 1), nn.Sigmoid())
        self.mod_s = nn.Sequential(nn.Conv2d(c_out, c_out, 1), nn.Sigmoid())
        self.out = nn.Sequential(nn.Conv2d(c_out, c_out, 3, padding=1, bias=False),
                                 nn.BatchNorm2d(c_out), nn.ReLU(inplace=True))

    def forward(self, f_cnn, f_swin):
        c = self.pc(f_cnn)
        s = self.ps(f_swin)
        if c.shape[-2:] != s.shape[-2:]:
            s = F.interpolate(s, size=c.shape[-2:], mode="bilinear", align_corners=False)
        # mutual modulation
        c2 = c * self.mod_s(s)   # context modulates texture
        s2 = s * self.mod_c(c)   # texture modulates context
        g = self.gate(torch.cat([c2, s2], dim=1))
        fused = g * c2 + (1.0 - g) * s2
        return self.out(fused) + fused


## 8. Dual encoders

Both encoders come from `timm` with `features_only=True`, returning four hierarchical stages. We auto-detect the per-stage channel counts (they differ slightly from the paper's figure depending on the timm version) and let the CMFG project each stage to the shared dims `[128, 256, 512, 1024]`. Swin returns channels-last maps in some timm versions, so we normalize everything to NCHW.

In [ ]:
# timm Swin may return channels-last maps (B, H, W, C). Detect and convert to (B, C, H, W).
_SHARED_DIMS = (128, 256, 512, 1024)
def _to_nchw(t):
    if t.dim() == 4 and t.shape[3] in _SHARED_DIMS and t.shape[1] not in _SHARED_DIMS:
        return t.permute(0, 3, 1, 2).contiguous()
    return t

class DualEncoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cnn = timm.create_model(cfg.cnn_backbone, pretrained=True,
                                      features_only=True, out_indices=(1, 2, 3, 4))
        self.swin = timm.create_model(cfg.swin_backbone, pretrained=True,
                                       features_only=True, out_indices=(0, 1, 2, 3),
                                       img_size=cfg.img_size)
        self.cnn_chs = self.cnn.feature_info.channels()
        self.swin_chs = self.swin.feature_info.channels()
        print("CNN stage channels :", self.cnn_chs)
        print("Swin stage channels:", self.swin_chs)

    def forward(self, x):
        cf = self.cnn(x)                       # 4 maps, NCHW
        sf = [_to_nchw(t) for t in self.swin(x)]  # 4 maps -> NCHW
        return cf, sf


## 9. Cascade attention decoder + full HieraCascadeNet

The fused pyramid `G¹..G⁴` (shared dims) feeds a U-Net-style cascade decoder:

- **Bottleneck** on `G⁴`: ASPP + strip pooling + CBAM → `decoder_dim` channels.
- Each **up-stage** upsamples, applies an **attention gate** to the corresponding fused skip, concatenates, convolves, and refines with **CBAM**. Intermediate stages emit **auxiliary segmentation maps** for deep supervision.
- **Segmentation head**: final 1-channel map at input resolution.
- **Classification head**: global-pooled bottleneck embedding → dropout → 7-way FC.


In [ ]:
class UpBlock(nn.Module):
    def __init__(self, c_in, c_skip, c_out):
        super().__init__()
        self.up = nn.ConvTranspose2d(c_in, c_out, 2, stride=2)
        self.gate = AttentionGate(c_out, c_skip, max(c_out // 2, 16))
        self.conv = nn.Sequential(
            nn.Conv2d(c_out + c_skip, c_out, 3, padding=1, bias=False),
            nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
            nn.Conv2d(c_out, c_out, 3, padding=1, bias=False),
            nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
        )
        self.cbam = CBAM(c_out)
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        skip = self.gate(x, skip)
        x = self.conv(torch.cat([x, skip], dim=1))
        return self.cbam(x)


class HieraCascadeNet(nn.Module):
    def __init__(self, cfg, num_classes=NUM_CLASSES):
        super().__init__()
        self.cfg = cfg
        self.encoder = DualEncoder(cfg)
        cnn_chs, swin_chs = self.encoder.cnn_chs, self.encoder.swin_chs
        fd = cfg.fused_dims  # [128,256,512,1024]

        # CMFG per stage
        self.cmfg = nn.ModuleList([
            CrossModalFusionGate(cnn_chs[i], swin_chs[i], fd[i]) for i in range(4)
        ])

        # bottleneck on deepest fused map G4 (fd[3])
        self.aspp = ASPP(fd[3], cfg.decoder_dim)
        self.strip = StripPooling(cfg.decoder_dim)
        self.bott_cbam = CBAM(cfg.decoder_dim)

        # decoder: skips are fd[2], fd[1], fd[0]
        self.up1 = UpBlock(cfg.decoder_dim, fd[2], 256)
        self.up2 = UpBlock(256, fd[1], 128)
        self.up3 = UpBlock(128, fd[0], 64)

        # final upsample to input resolution + seg head
        self.up_final = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 2, stride=2),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
        )
        self.seg_head = nn.Conv2d(32, 1, 1)

        # deep-supervision heads (auxiliary seg from intermediate decoder levels)
        self.aux1 = nn.Conv2d(256, 1, 1)
        self.aux2 = nn.Conv2d(128, 1, 1)

        # classification head from bottleneck embedding
        self.cls_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(cfg.decoder_dim, 512), nn.ReLU(inplace=True), nn.Dropout(0.4),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        H, W = x.shape[-2:]
        cf, sf = self.encoder(x)
        G = [self.cmfg[i](cf[i], sf[i]) for i in range(4)]  # G1..G4

        b = self.aspp(G[3])
        b = self.strip(b)
        b = self.bott_cbam(b)                 # bottleneck (decoder_dim, 7x7)

        d1 = self.up1(b, G[2])                 # -> 14x14, 256
        d2 = self.up2(d1, G[1])               # -> 28x28, 128
        d3 = self.up3(d2, G[0])               # -> 56x56, 64
        d4 = self.up_final(d3)                # -> 112x112, 32
        seg = self.seg_head(d4)
        seg = F.interpolate(seg, size=(H, W), mode="bilinear", align_corners=False)

        logits = self.cls_head(b)

        aux1 = F.interpolate(self.aux1(d1), size=(H, W), mode="bilinear", align_corners=False)
        aux2 = F.interpolate(self.aux2(d2), size=(H, W), mode="bilinear", align_corners=False)

        return {"logits": logits, "seg": seg, "aux": [aux1, aux2], "embed": b}

    # helpers for two-stage training
    def freeze_encoders(self):
        for p in self.encoder.parameters():
            p.requires_grad = False
    def unfreeze_encoders(self):
        for p in self.encoder.parameters():
            p.requires_grad = True


### Quick shape sanity check
Instantiate the model and run a random batch to confirm all shapes line up (works without the dataset).

In [ ]:
model = HieraCascadeNet(cfg).to(DEVICE)
model.eval()
with torch.no_grad():
    dummy = torch.randn(2, 3, cfg.img_size, cfg.img_size, device=DEVICE)
    out = model(dummy)
print("logits:", tuple(out["logits"].shape))
print("seg   :", tuple(out["seg"].shape))
print("aux   :", [tuple(a.shape) for a in out["aux"]])
print("embed :", tuple(out["embed"].shape))
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"params: {n_params:.1f} M")

## 10. Losses

The combined objective is `L_total = L_cls + λ1·L_seg + λ2·L_ds`:

- **`L_cls`** — focal cross-entropy (label-smoothed) to focus on hard/ambiguous lesions.
- **`L_seg`** — Dice + BCE on the main segmentation output (stable under the large background:lesion ratio). Applied only to samples that actually have a mask.
- **`L_ds`** — deep-supervision Dice on the auxiliary decoder outputs.

If a batch has no masks, the seg/ds terms drop to zero automatically and the model trains as a pure classifier.

In [ ]:
class FocalCE(nn.Module):
    def __init__(self, gamma=2.0, smoothing=0.05, weight=None):
        super().__init__()
        self.gamma = gamma
        self.smoothing = smoothing
        self.weight = weight
    def forward(self, logits, target):
        ce = F.cross_entropy(logits, target, weight=self.weight,
                             label_smoothing=self.smoothing, reduction="none")
        pt = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()

def dice_loss(logits, target, eps=1.0):
    prob = torch.sigmoid(logits)
    prob = prob.flatten(1); target = target.flatten(1)
    inter = (prob * target).sum(1)
    denom = prob.sum(1) + target.sum(1)
    return (1 - (2 * inter + eps) / (denom + eps)).mean()

def seg_loss_fn(seg_logits, mask, has_mask):
    m = has_mask.view(-1) > 0.5
    if m.sum() == 0:
        return seg_logits.sum() * 0.0  # keep graph, zero contribution
    sl, mm = seg_logits[m], mask[m]
    bce = F.binary_cross_entropy_with_logits(sl, mm)
    return 0.5 * bce + 0.5 * dice_loss(sl, mm)

class HieraLoss(nn.Module):
    def __init__(self, cfg, class_weight=None):
        super().__init__()
        self.cfg = cfg
        self.cls = FocalCE(cfg.focal_gamma, cfg.label_smoothing, class_weight)
    def forward(self, out, batch):
        l_cls = self.cls(out["logits"], batch["label"])
        l_seg = seg_loss_fn(out["seg"], batch["mask"], batch["has_mask"])
        l_ds = 0.0
        for a in out["aux"]:
            l_ds = l_ds + seg_loss_fn(a, batch["mask"], batch["has_mask"])
        l_ds = l_ds / max(len(out["aux"]), 1)
        total = l_cls + self.cfg.lambda_seg * l_seg + self.cfg.lambda_ds * l_ds
        return total, {"cls": float(l_cls), "seg": float(l_seg), "ds": float(l_ds)}


## 11. Training utilities

One epoch of training / validation, plus a metric helper. Uses AMP, gradient clipping, and returns loss + accuracy + macro-F1.

In [ ]:
from sklearn.metrics import f1_score, accuracy_score
from tqdm.auto import tqdm

def run_epoch(model, loader, criterion, optimizer=None, scaler=None, cfg=None):
    train = optimizer is not None
    model.train(train)
    tot_loss, ys, ps = 0.0, [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in tqdm(loader, leave=False):
            imgs = batch["image"].to(DEVICE, non_blocking=True)
            batch = {k: (v.to(DEVICE, non_blocking=True) if torch.is_tensor(v) else v)
                     for k, v in batch.items()}
            if train:
                optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=cfg.amp and DEVICE.type == "cuda"):
                out = model(imgs)
                loss, _ = criterion(out, batch)
            if train:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
                scaler.step(optimizer); scaler.update()
            tot_loss += float(loss) * imgs.size(0)
            ys.append(batch["label"].cpu().numpy())
            ps.append(out["logits"].argmax(1).cpu().numpy())
    import numpy as np
    ys = np.concatenate(ys); ps = np.concatenate(ps)
    return {"loss": tot_loss / len(loader.dataset),
            "acc": accuracy_score(ys, ps),
            "macro_f1": f1_score(ys, ps, average="macro")}


## 12. Two-stage training

**Stage 1** — freeze both encoders, warm up the fusion/decoder/heads with a higher LR.
**Stage 2** — unfreeze everything and fine-tune end-to-end with a low LR (separate param group for backbones). Cosine LR, early stopping on validation macro-F1, best checkpoint saved.

In [ ]:
def build_optimizer(model, cfg, stage):
    if stage == "frozen":
        params = [p for p in model.parameters() if p.requires_grad]
        return torch.optim.AdamW(params, lr=cfg.lr_frozen, weight_decay=cfg.weight_decay)
    # finetune: lower LR for backbone, slightly higher for the rest
    enc_ids = set(id(p) for p in model.encoder.parameters())
    backbone = [p for p in model.parameters() if id(p) in enc_ids and p.requires_grad]
    head = [p for p in model.parameters() if id(p) not in enc_ids and p.requires_grad]
    return torch.optim.AdamW([
        {"params": backbone, "lr": cfg.lr_backbone},
        {"params": head, "lr": cfg.lr_finetune},
    ], weight_decay=cfg.weight_decay)


def train_hieracascadenet(model, dl_train, dl_val, cfg, class_weight=None):
    criterion = HieraLoss(cfg, class_weight).to(DEVICE)
    scaler = torch.cuda.amp.GradScaler(enabled=cfg.amp and DEVICE.type == "cuda")
    history, best_f1, best_path, patience = [], -1.0, os.path.join(cfg.out_dir, "best.pt"), 0

    def loop(n_epochs, stage, start_ep=0):
        nonlocal best_f1, patience
        optimizer = build_optimizer(model, cfg, stage)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
        for ep in range(n_epochs):
            tr = run_epoch(model, dl_train, criterion, optimizer, scaler, cfg)
            va = run_epoch(model, dl_val, criterion, None, None, cfg)
            sched.step()
            row = {"epoch": start_ep + ep, "stage": stage,
                   "train_loss": tr["loss"], "val_loss": va["loss"],
                   "train_acc": tr["acc"], "val_acc": va["acc"],
                   "train_f1": tr["macro_f1"], "val_f1": va["macro_f1"]}
            history.append(row)
            print(f"[{stage}] ep{start_ep+ep:03d} "
                  f"train_loss {tr['loss']:.4f} val_loss {va['loss']:.4f} "
                  f"val_acc {va['acc']:.4f} val_f1 {va['macro_f1']:.4f}")
            if va["macro_f1"] > best_f1:
                best_f1 = va["macro_f1"]; patience = 0
                torch.save(model.state_dict(), best_path)
            else:
                patience += 1
                if patience >= cfg.early_stop_patience:
                    print("early stopping."); return True
        return False

    # Stage 1
    model.freeze_encoders()
    stopped = loop(cfg.epochs_frozen, "frozen", 0)
    # Stage 2
    if not stopped:
        model.unfreeze_encoders()
        patience = 0
        loop(cfg.epochs_finetune, "finetune", cfg.epochs_frozen)

    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    print(f"best val macro-F1: {best_f1:.4f} (restored {best_path})")
    return pd.DataFrame(history)


### Optional: class weights for the loss

Even with a balanced sampler, mild inverse-frequency class weights in the focal-CE can help the rarest classes. Computed from the training split.

In [ ]:
def compute_class_weights(frame):
    import numpy as np
    counts = np.bincount(frame["label"].values, minlength=NUM_CLASSES).astype(np.float64)
    counts[counts == 0] = 1.0
    w = counts.sum() / (NUM_CLASSES * counts)
    w = w / w.mean()
    return torch.tensor(w, dtype=torch.float32, device=DEVICE)

# Launch training (requires the dataset). Uncomment to run:
# class_w = compute_class_weights(df_train)
# history = train_hieracascadenet(model, dl_train, dl_val, cfg, class_weight=class_w)
# history.to_csv(os.path.join(cfg.out_dir, "history.csv"), index=False)

## 13. Training curves

Plot loss/accuracy across the frozen → fine-tune transition (paper Fig. 6).

In [ ]:
import matplotlib.pyplot as plt

def plot_curves(history, cfg):
    if isinstance(history, str):
        history = pd.read_csv(history)
    switch = history[history["stage"] == "finetune"]["epoch"].min()
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ax[0].plot(history["epoch"], history["train_loss"], label="train")
    ax[0].plot(history["epoch"], history["val_loss"], label="val")
    ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].plot(history["epoch"], history["train_acc"], label="train")
    ax[1].plot(history["epoch"], history["val_acc"], label="val")
    ax[1].set_title("Accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend()
    if pd.notna(switch):
        for a in ax:
            a.axvline(switch, ls="--", color="green", alpha=0.6)
    plt.tight_layout()
    plt.savefig(os.path.join(cfg.out_dir, "training_curves.png"), dpi=150, bbox_inches="tight")
    plt.show()

# plot_curves(history, cfg)

## 14. Evaluation on the test set

Collects logits/probabilities over the test loader, then computes: overall accuracy, macro precision/recall/F1, per-class F1, confusion matrix, one-vs-rest ROC + AUC, and precision–recall + AP — the full panel from the paper.

In [ ]:
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, precision_recall_curve,
                             average_precision_score)
import seaborn as sns

@torch.no_grad()
def collect_predictions(model, loader):
    model.eval()
    all_logits, all_labels, all_embed = [], [], []
    for batch in tqdm(loader, leave=False):
        imgs = batch["image"].to(DEVICE)
        out = model(imgs)
        all_logits.append(out["logits"].cpu())
        all_labels.append(batch["label"])
        all_embed.append(F.adaptive_avg_pool2d(out["embed"], 1).flatten(1).cpu())
    logits = torch.cat(all_logits); labels = torch.cat(all_labels)
    probs = torch.softmax(logits, 1).numpy()
    embed = torch.cat(all_embed).numpy()
    return probs, labels.numpy(), embed

def evaluate(model, loader, cfg):
    probs, y, embed = collect_predictions(model, loader)
    preds = probs.argmax(1)
    from sklearn.preprocessing import label_binarize
    y_bin = label_binarize(y, classes=list(range(NUM_CLASSES)))

    print(classification_report(y, preds, target_names=CLASSES, digits=4))
    acc = accuracy_score(y, preds)
    macro_f1 = f1_score(y, preds, average="macro")
    macro_auc = roc_auc_score(y_bin, probs, average="macro", multi_class="ovr")
    micro_auc = roc_auc_score(y_bin, probs, average="micro", multi_class="ovr")
    macro_ap = average_precision_score(y_bin, probs, average="macro")
    print(f"accuracy={acc:.4f}  macroF1={macro_f1:.4f}  "
          f"macroAUC={macro_auc:.4f}  microAUC={micro_auc:.4f}  macroAP={macro_ap:.4f}")
    return {"probs": probs, "y": y, "embed": embed, "y_bin": y_bin,
            "acc": acc, "macro_f1": macro_f1, "macro_auc": macro_auc, "macro_ap": macro_ap}

# res = evaluate(model, dl_test, cfg)

### Confusion matrix (Fig. 3)

In [ ]:
def plot_confusion(res, cfg, normalize=True):
    cm = confusion_matrix(res["y"], res["probs"].argmax(1))
    cmn = cm.astype(float) / cm.sum(1, keepdims=True) if normalize else cm
    plt.figure(figsize=(7, 6))
    sns.heatmap(cmn, annot=True, fmt=".2f" if normalize else "d", cmap="Greens",
                xticklabels=CLASSES, yticklabels=CLASSES, cbar=True)
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.title(f"Confusion matrix (overall acc {res['acc']*100:.2f}%)")
    plt.tight_layout()
    plt.savefig(os.path.join(cfg.out_dir, "confusion_matrix.png"), dpi=150, bbox_inches="tight")
    plt.show()

# plot_confusion(res, cfg)

### ROC and precision–recall curves (Figs. 4–5)

In [ ]:
def plot_roc_pr(res, cfg):
    y_bin, probs = res["y_bin"], res["probs"]
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    for i, c in enumerate(CLASSES):
        fpr, tpr, _ = roc_curve(y_bin[:, i], probs[:, i])
        auc_i = roc_auc_score(y_bin[:, i], probs[:, i])
        ax[0].plot(fpr, tpr, label=f"{c} (AUC={auc_i:.3f})")
        pr, rc, _ = precision_recall_curve(y_bin[:, i], probs[:, i])
        ap_i = average_precision_score(y_bin[:, i], probs[:, i])
        ax[1].plot(rc, pr, label=f"{c} (AP={ap_i:.3f})")
    ax[0].plot([0, 1], [0, 1], "k--", alpha=0.4)
    ax[0].set_title("ROC (one-vs-rest)"); ax[0].set_xlabel("FPR"); ax[0].set_ylabel("TPR"); ax[0].legend(fontsize=8)
    ax[1].set_title("Precision–Recall"); ax[1].set_xlabel("Recall"); ax[1].set_ylabel("Precision"); ax[1].legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(cfg.out_dir, "roc_pr_curves.png"), dpi=150, bbox_inches="tight")
    plt.show()

# plot_roc_pr(res, cfg)

## 15. Interpretability — t-SNE of penultimate embeddings (Fig. 7)

Project the pooled bottleneck embeddings to 2-D to check class separability.

In [ ]:
def plot_tsne(res, cfg, max_pts=3000):
    from sklearn.manifold import TSNE
    import numpy as np
    emb, y = res["embed"], res["y"]
    if len(emb) > max_pts:
        idx = np.random.RandomState(0).choice(len(emb), max_pts, replace=False)
        emb, y = emb[idx], y[idx]
    z = TSNE(n_components=2, perplexity=30, init="pca", random_state=0).fit_transform(emb)
    plt.figure(figsize=(8, 7))
    for i, c in enumerate(CLASSES):
        m = y == i
        plt.scatter(z[m, 0], z[m, 1], s=8, label=c, alpha=0.6)
    plt.legend(markerscale=2); plt.title("t-SNE of penultimate embeddings")
    plt.tight_layout()
    plt.savefig(os.path.join(cfg.out_dir, "tsne.png"), dpi=150, bbox_inches="tight")
    plt.show()

# plot_tsne(res, cfg)

## 16. Interpretability — Grad-CAM (Fig. 8)

Grad-CAM on the final convolutional stage of the EfficientNet-B4 branch, w.r.t. the predicted class. A lightweight hook-based implementation (no external dependency).

In [ ]:
class GradCAM:
    def __init__(self, model, target_module):
        self.model = model
        self.acts = None; self.grads = None
        target_module.register_forward_hook(self._fwd)
        target_module.register_full_backward_hook(self._bwd)
    def _fwd(self, m, i, o): self.acts = o.detach()
    def _bwd(self, m, gi, go): self.grads = go[0].detach()
    def __call__(self, x, class_idx=None):
        self.model.eval()
        out = self.model(x)
        logits = out["logits"]
        if class_idx is None:
            class_idx = logits.argmax(1)
        score = logits.gather(1, class_idx.view(-1, 1)).sum()
        self.model.zero_grad(); score.backward(retain_graph=True)
        w = self.grads.mean(dim=(2, 3), keepdim=True)      # GAP over gradients
        cam = F.relu((w * self.acts).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)
        cam = cam - cam.amin(dim=(2, 3), keepdim=True)
        cam = cam / (cam.amax(dim=(2, 3), keepdim=True) + 1e-6)
        return cam.cpu().numpy(), class_idx.cpu().numpy(), torch.softmax(logits,1).detach().cpu().numpy()

def denorm(t):
    import numpy as np
    mean = np.array([0.485, 0.456, 0.406]); std = np.array([0.229, 0.224, 0.225])
    img = t.cpu().numpy().transpose(1, 2, 0) * std + mean
    return np.clip(img, 0, 1)

def show_gradcam(model, loader, cfg, n=4):
    import numpy as np
    # last conv stage of EfficientNet branch
    target = model.encoder.cnn
    cam = GradCAM(model, target)
    batch = next(iter(loader))
    imgs = batch["image"][:n].to(DEVICE)
    maps, cls, probs = cam(imgs, None)
    fig, ax = plt.subplots(n, 3, figsize=(9, 3 * n))
    if n == 1: ax = ax[None, :]
    for k in range(n):
        base = denorm(imgs[k])
        ax[k, 0].imshow(base); ax[k, 0].set_title("input"); ax[k, 0].axis("off")
        ax[k, 1].imshow(maps[k, 0], cmap="jet"); ax[k, 1].set_title("Grad-CAM"); ax[k, 1].axis("off")
        ax[k, 2].imshow(base); ax[k, 2].imshow(maps[k, 0], cmap="jet", alpha=0.45)
        p = probs[k, cls[k]]
        ax[k, 2].set_title(f"{CLASSES[cls[k]]} ({p*100:.1f}%)"); ax[k, 2].axis("off")
    plt.tight_layout()
    plt.savefig(os.path.join(cfg.out_dir, "gradcam.png"), dpi=150, bbox_inches="tight")
    plt.show()

# show_gradcam(model, dl_test, cfg, n=4)

## 17. Computational complexity (Table III)

Report parameters, FLOPs, model size, latency and throughput. FLOPs use `ptflops` if available; otherwise only params/latency are reported.

In [ ]:
def profile_model(model, cfg, iters=50):
    import numpy as np
    model.eval()
    n_params = sum(p.numel() for p in model.parameters())
    size_mb = sum(p.numel() * p.element_size() for p in model.parameters()) / 1e6
    flops = None
    try:
        from ptflops import get_model_complexity_info
        class W(nn.Module):
            def __init__(s, m): super().__init__(); s.m = m
            def forward(s, x): return s.m(x)["logits"]
        macs, _ = get_model_complexity_info(W(model), (3, cfg.img_size, cfg.img_size),
                                            as_strings=False, print_per_layer_stat=False, verbose=False)
        flops = 2 * macs / 1e9  # GFLOPs
    except Exception as e:
        print("[info] ptflops unavailable:", e)

    x = torch.randn(1, 3, cfg.img_size, cfg.img_size, device=DEVICE)
    with torch.no_grad():
        for _ in range(5): model(x)
        if DEVICE.type == "cuda": torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(iters): model(x)
        if DEVICE.type == "cuda": torch.cuda.synchronize()
        lat = (time.time() - t0) / iters * 1000  # ms
    print(f"params={n_params/1e6:.1f}M  size={size_mb:.0f}MB  "
          f"GFLOPs={flops if flops is None else round(flops,1)}  "
          f"latency={lat:.1f}ms  throughput={1000/lat:.0f} img/s")
    return {"params_M": n_params/1e6, "size_MB": size_mb, "GFLOPs": flops,
            "latency_ms": lat, "throughput": 1000/lat}

# profile_model(model, cfg)

## 18. Ablation & comparison scaffolding (Tables I–II)

The paper's ablation adds components one at a time. This notebook's `HieraCascadeNet` is the *full* model; to reproduce the ablation rows you toggle components and retrain. Below is a thin config-driven switchboard you can extend — set flags, rebuild the model, and reuse `train_hieracascadenet` + `evaluate`.

| Row | fusion | cbam/attn skips | aspp+strip | multi-task+deep-sup |
|-----|:------:|:---------------:|:----------:|:-------------------:|
| EfficientNet-B4 only | – | – | – | – |
| Swin-Base only | – | – | – | – |
| Dual, naïve concat | concat | – | – | – |
| + CMFG | gate | – | – | – |
| + CBAM & attn skips | gate | ✓ | – | – |
| + ASPP & strip | gate | ✓ | ✓ | – |
| Full | gate | ✓ | ✓ | ✓ |

The simplest, robust way to run true single-backbone rows is to train a plain `timm` classifier for the `EfficientNet-B4 only` / `Swin-Base only` baselines, and set `cfg.lambda_seg = cfg.lambda_ds = 0` (and `use_class_balanced_sampler` as desired) for the no-multi-task rows.

In [ ]:
def build_single_backbone_baseline(backbone, num_classes=NUM_CLASSES):
    """Baseline classifier for ablation rows 1-2 (EfficientNet-B4 only / Swin-Base only)."""
    m = timm.create_model(backbone, pretrained=True, num_classes=num_classes)
    return m.to(DEVICE)

# Example (uncomment):
# eff = build_single_backbone_baseline("efficientnet_b4")
# swin = build_single_backbone_baseline("swin_base_patch4_window7_224")
# (wrap with a small train loop reusing FocalCE + the balanced sampler)

## 19. One-call end-to-end run

With the dataset in place, this is the whole pipeline start to finish.

In [ ]:
def run_all(cfg):
    df = build_dataframe(cfg)
    df_train, df_val, df_test = stratified_group_split(df, cfg)
    dl_train, dl_val, dl_test = make_loaders(cfg, df_train, df_val, df_test)

    model = HieraCascadeNet(cfg).to(DEVICE)
    class_w = compute_class_weights(df_train)
    history = train_hieracascadenet(model, dl_train, dl_val, cfg, class_weight=class_w)
    history.to_csv(os.path.join(cfg.out_dir, "history.csv"), index=False)

    plot_curves(history, cfg)
    res = evaluate(model, dl_test, cfg)
    plot_confusion(res, cfg)
    plot_roc_pr(res, cfg)
    plot_tsne(res, cfg)
    show_gradcam(model, dl_test, cfg)
    profile_model(model, cfg)
    return model, history, res

# model, history, res = run_all(cfg)

## 20. Self-contained smoke test (no dataset, no internet)

Validates that all custom blocks and the decoder wire together correctly on random tensors, using **stub encoders** so it runs even without downloading `timm` pretrained weights. This is what to run first to confirm the architecture is sound before pointing it at HAM10000.

In [ ]:
def _smoke_test():
    class StubEncoder(nn.Module):
        """Mimics DualEncoder outputs without any pretrained download."""
        def __init__(self):
            super().__init__()
            self.cnn_chs = [32, 56, 112, 272]
            self.swin_chs = [128, 256, 512, 1024]
            # tiny conv stems to produce 4 pyramid levels at /4 /8 /16 /32
            self.cnn_proj = nn.ModuleList([nn.Conv2d(3, c, 1) for c in self.cnn_chs])
            self.swin_proj = nn.ModuleList([nn.Conv2d(3, c, 1) for c in self.swin_chs])
        def forward(self, x):
            sizes = [56, 28, 14, 7]
            cf = [self.cnn_proj[i](F.interpolate(x, size=s)) for i, s in enumerate(sizes)]
            sf = [self.swin_proj[i](F.interpolate(x, size=s)) for i, s in enumerate(sizes)]
            return cf, sf

    net = HieraCascadeNet.__new__(HieraCascadeNet)   # bypass __init__ (which loads timm)
    nn.Module.__init__(net)
    net.cfg = cfg
    net.encoder = StubEncoder()
    fd = cfg.fused_dims
    net.cmfg = nn.ModuleList([CrossModalFusionGate(net.encoder.cnn_chs[i],
                                                   net.encoder.swin_chs[i], fd[i]) for i in range(4)])
    net.aspp = ASPP(fd[3], cfg.decoder_dim)
    net.strip = StripPooling(cfg.decoder_dim)
    net.bott_cbam = CBAM(cfg.decoder_dim)
    net.up1 = UpBlock(cfg.decoder_dim, fd[2], 256)
    net.up2 = UpBlock(256, fd[1], 128)
    net.up3 = UpBlock(128, fd[0], 64)
    net.up_final = nn.Sequential(nn.ConvTranspose2d(64, 32, 2, stride=2),
                                 nn.BatchNorm2d(32), nn.ReLU(inplace=True))
    net.seg_head = nn.Conv2d(32, 1, 1)
    net.aux1 = nn.Conv2d(256, 1, 1)
    net.aux2 = nn.Conv2d(128, 1, 1)
    net.cls_head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                 nn.Linear(cfg.decoder_dim, 512), nn.ReLU(inplace=True),
                                 nn.Dropout(0.4), nn.Linear(512, NUM_CLASSES))
    net = net.to(DEVICE)

    x = torch.randn(2, 3, cfg.img_size, cfg.img_size, device=DEVICE)
    out = net(x)
    assert out["logits"].shape == (2, NUM_CLASSES)
    assert out["seg"].shape == (2, 1, cfg.img_size, cfg.img_size)
    assert all(a.shape == (2, 1, cfg.img_size, cfg.img_size) for a in out["aux"])

    # loss + one backward step
    crit = HieraLoss(cfg).to(DEVICE)
    batch = {"label": torch.randint(0, NUM_CLASSES, (2,), device=DEVICE),
             "mask": (torch.rand(2, 1, cfg.img_size, cfg.img_size, device=DEVICE) > 0.5).float(),
             "has_mask": torch.ones(2, device=DEVICE)}
    loss, parts = crit(out, batch)
    loss.backward()
    print("smoke test passed ✓ | loss %.4f | parts %s" % (float(loss), {k: round(v,4) for k,v in parts.items()}))
    print("logits", tuple(out["logits"].shape), "| seg", tuple(out["seg"].shape))

_smoke_test()

## 21. Notes for GitHub

**Dataset layout expected** (`cfg.data_root`):
```
data/ham10000/
├── HAM10000_metadata.csv
├── HAM10000_images_part_1/*.jpg
├── HAM10000_images_part_2/*.jpg
└── HAM10000_segmentations_lesion_tschandl/*_segmentation.png   # optional (for the seg branch)
```

**How to run**
1. Run cells 0–1 (env + config); edit `cfg.data_root`.
2. Run cell 20 (**smoke test**) — verifies wiring with no dataset/internet.
3. Point `cfg` at HAM10000 and run cell 19 (`run_all`) end-to-end, or run sections 2–17 individually.

**Reproducibility caveats.** Exact paper numbers depend on the specific HAM10000 download, mask availability, near-duplicate filtering, seeds, and GPU. The class-balanced *test* set the paper reports (4,694 imgs) is balanced by oversampling on the test partition; if you evaluate on the natural (imbalanced) test set instead, macro metrics will differ. Keep oversampling strictly on the training split for a leakage-free, honest estimate, and additionally report metrics on the natural test distribution.

**License / attribution.** Cite the HAM10000 dataset (Tschandl et al., 2018) and the backbones (EfficientNet, Swin Transformer) per their licenses. This code re-implements the described method; verify against the original paper before publishing results.
